# Notebook B: chia dữ liệu, mô hình và câu hỏi thứ ba

**Đề tài:** Cảnh báo nước biển không an toàn cho người tắm từ lượng mưa và lịch sử điểm đo tại bảy mươi chín bãi ở Sydney trong ba thập kỷ

**Sinh viên B** chạy sau khi Notebook A đã tạo file parquet, `sql/queries.sql` và `manifest.json`. Notebook này không cần kernel của Notebook A.

## Kiểm tra bàn giao

- **Làm gì:** đọc manifest và so mã băm của file parquet.
- **Vì sao:** bảo đảm hai notebook dùng đúng cùng một dữ liệu.
- **Kiểm tra:** ô in `handover ok`.

In [ ]:
# run once per machine, then comment out
# %pip install pandas numpy duckdb pyarrow scikit-learn lightgbm xgboost matplotlib scipy
import os, json, hashlib, numpy as np, pandas as pd, warnings
warnings.filterwarnings('ignore')
m = json.load(open('data/processed/manifest.json'))
assert hashlib.md5(open(m['file'], 'rb').read()).hexdigest() == m['md5'], 'parquet differs from the manifest, rerun Notebook A'
os.makedirs('report', exist_ok=True); print('handover ok', m['rows'], 'rows')

## Bước 4. Bảng đặc trưng và chia dữ liệu

- **Làm gì:** chạy lại `sql/queries.sql`, lấy bảng `feat`, chia tập huấn luyện và kiểm thử theo giao thức của đề tài.
- **Vì sao:** cách chia quyết định con số có đáng tin hay không; dòng `assert` là phép kiểm tra rò rỉ.
- **Kiểm tra:** ghi số dòng hai tập và số đặc trưng vào bảng mô tả thực nghiệm.

In [ ]:
# Step 4: forward split: samples before 2020 for training, samples from 2020 to 2025 for testing; 30% of the sites are also kept out of training for RQ3
import duckdb
con = duckdb.connect(); con.execute(open('sql/queries.sql').read())
feat = con.execute("SELECT * FROM feat ORDER BY row_id").df(); feat['date'] = pd.to_datetime(feat.date)
RAIN_F = ['rain_today', 'rain_1d', 'rain_3d', 'rain_7d', 'air_temp_max', 'month_sin', 'month_cos']; SITE_F = ['site_unsafe_rate_before', 'last_log_count', 'days_since_last_sample', 'latitude', 'longitude', 'region_code']; PROBE_F = ['water_temp', 'conductivity', 'temp_missing', 'cond_missing']
X_cols = RAIN_F + SITE_F + PROBE_F; TARGET = 'unsafe'
old = feat[feat.year < 2020]; med = old[['water_temp', 'conductivity']].median(); feat[['water_temp', 'conductivity']] = feat[['water_temp', 'conductivity']].fillna(med)      # medians of the training years only
sites = np.sort(feat.site.unique()); UNSEEN = set(np.random.RandomState(0).choice(sites, int(0.3 * len(sites)), replace=False)); cap = lambda d, n: d.sample(min(len(d), n), random_state=0)
train = cap(feat[(feat.year < 2020) & ~feat.site.isin(UNSEEN)], 60_000); test = feat[(feat.year >= 2020) & ~feat.site.isin(UNSEEN)]; test_unseen = feat[(feat.year >= 2020) & feat.site.isin(UNSEEN)]
X_tr, y_tr, X_te, y_te = train[X_cols], train[TARGET], test[X_cols], test[TARGET]
assert train.date.max() < test.date.min() and not set(train.site) & UNSEEN       # leakage checks
print(len(feat), len(train), len(test), len(test_unseen), len(X_cols), 'features', len(sites), 'sites', len(UNSEEN), 'held out', 'unsafe share train and test', round(y_tr.mean(), 3), round(y_te.mean(), 3))

## Bước 4b. Hiểu tập huấn luyện và tập kiểm thử

Ba ô dưới đây kiểm tra hai tập có giống nhau không và đặc trưng nào mạnh nhất, trước khi huấn luyện bất kỳ mô hình nào.

### 4b.1 Mục tiêu ở hai tập

- **Làm gì:** so phân phối của mục tiêu giữa tập huấn luyện và tập kiểm thử.
- **Vì sao:** hai phân phối lệch nhau giải thích vì sao mô hình cây ngoại suy kém.

> **Nhận xét của nhóm (bắt buộc, một câu có số):** ...

In [ ]:
import matplotlib.pyplot as plt
def bar_chart(labels, values, ylabel, path, highlight=None, fmt='{:.1f}'):
    fig, ax = plt.subplots(figsize=(6.5, 3.2))
    colors = ['#F4A261' if i == highlight else '#1B6B6D' for i in range(len(values))]
    bars = ax.bar([str(l) for l in labels], values, color=colors)
    ax.bar_label(bars, labels=[fmt.format(v) for v in values], fontweight='bold', padding=2, fontsize=8 if len(values) > 7 else 10)
    ax.set_ylabel(ylabel); ax.set_ylim(0, max(values) * 1.2)
    ax.spines[['top', 'right']].set_visible(False)          # no top and right spines
    if len(values) > 7: ax.tick_params(axis='x', labelsize=7, rotation=30); plt.setp(ax.get_xticklabels(), ha='right')
    fig.tight_layout(); fig.savefig(path, dpi=300); plt.show()

import matplotlib.pyplot as plt
def nice(v, label):                                     # readable numbers on the bars
    v = np.asarray(v, dtype=float); m = np.nanmax(np.abs(v))
    return (v / 1e6, label + ' (million)', '{:.2f}') if m >= 1e6 else (v / 1e3, label + ' (thousand)', '{:.1f}') if m >= 1e4 else (v, label, '{:.2f}' if m < 100 else '{:.0f}')
ytr, yte = pd.Series(np.asarray(y_tr, dtype=float)), pd.Series(np.asarray(y_te, dtype=float)); BINARY = ytr.nunique() <= 2
print('train rows', len(ytr), 'test rows', len(yte), 'features', X_tr.shape[1])
if BINARY: print('positive share: train %.3f, test %.3f' % (ytr.mean(), yte.mean()))
else:
    fig, ax = plt.subplots(figsize=(6.5, 3)); lo, hi = min(ytr.quantile(.01), yte.quantile(.01)), max(ytr.quantile(.99), yte.quantile(.99)); bins = np.linspace(lo, hi, 50)
    ax.hist(ytr.clip(lo, hi), bins=bins, density=True, alpha=.6, color='#1B6B6D', label='Train'); ax.hist(yte.clip(lo, hi), bins=bins, density=True, alpha=.6, color='#F4A261', label='Test')
    ax.legend(frameon=False); ax.set_xlabel('Target'); ax.set_ylabel('Density'); ax.spines[['top', 'right']].set_visible(False); fig.tight_layout(); fig.savefig('report/fig_split_target.png', dpi=200); plt.show()
    print(pd.DataFrame({'train': ytr.describe(), 'test': yte.describe()}).round(3))

### 4b.2 Đặc trưng lệch giữa hai tập

- **Làm gì:** độ lệch trung bình chuẩn hoá của từng đặc trưng giữa hai tập.
- **Kiểm tra:** đặc trưng lệch trên 0,5 độ lệch chuẩn là nguồn rủi ro, nêu trong mục Limitations.

> **Nhận xét của nhóm (bắt buộc, một câu có số):** ...

In [ ]:
A_, B_ = X_tr.astype(float), X_te.astype(float); sd = A_.std().replace(0, np.nan)
shift = ((B_.mean() - A_.mean()) / sd).abs().sort_values(ascending=False).rename('abs_standardised_mean_difference')
shift.to_csv('report/table_split_shift.csv'); display(shift.head(10).round(3).to_frame())
print('features whose test mean lies more than 0.5 training standard deviations away:', int((shift > 0.5).sum()), 'of', len(shift))

### 4b.3 Sức mạnh của từng đặc trưng

- **Làm gì:** tương quan hạng giữa từng đặc trưng và mục tiêu trên tập huấn luyện.
- **Vì sao:** nếu một đặc trưng đã gần đủ, mô hình phức tạp sẽ không hơn nhiều, và đó là thông tin để chọn câu hỏi.

> **Nhận xét của nhóm (bắt buộc, một câu có số):** ...

In [ ]:
sig = A_.sample(min(len(A_), 20_000), random_state=0); ys = pd.Series(np.asarray(y_tr, dtype=float), index=X_tr.index).loc[sig.index]
strength = sig.corrwith(ys, method='spearman').abs().sort_values(ascending=False).rename('abs_rank_correlation_with_target')
strength.to_csv('report/table_feature_strength.csv'); display(strength.head(10).round(3).to_frame())
print('strongest single feature:', strength.index[0], round(float(strength.iloc[0]), 3), '-> a model must beat what this feature already gives')

## Bước 6a. So sánh mô hình (RQ2)

- **Làm gì:** huấn luyện mười mô hình với năm seed trên cùng cách chia.
- **Vì sao:** so nhiều họ mô hình để kết luận không phụ thuộc một kiểu mô hình; năm seed cho độ lệch chuẩn.
- **Kiểm tra:** chênh lệch nhỏ hơn độ lệch chuẩn thì chưa được kết luận mô hình nào hơn. Bảng lưu ở `report/table_rq2.csv`.

In [ ]:
# Step 6a: ten classifiers from four families, five seeds, trained on samples before 2020 and tested on 2020 to 2025, unsafe against safe samples; probability metrics because one class is rare
import numpy as np, pandas as pd, time
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
sc = lambda m: make_pipeline(StandardScaler(), m)       # scaling for linear, distance and neural models
models = {
    'Logistic':       lambda s: sc(LogisticRegression(max_iter=500, random_state=s)),
    'Linear SVM':     lambda s: sc(LinearSVC(C=1.0, random_state=s)),
    'k-NN':           lambda s: sc(KNeighborsClassifier(n_neighbors=15)),
    'Decision Tree':  lambda s: DecisionTreeClassifier(max_depth=12, min_samples_leaf=5, random_state=s),
    'Random Forest':  lambda s: RandomForestClassifier(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=s),
    'Extra Trees':    lambda s: ExtraTreesClassifier(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=s),
    'HistGB':         lambda s: HistGradientBoostingClassifier(max_iter=200, learning_rate=0.1, random_state=s),
    'LightGBM':       lambda s: LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31, subsample=0.8,
                                               subsample_freq=1, colsample_bytree=0.8, random_state=s, verbose=-1),
    'XGBoost':        lambda s: XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6, subsample=0.8,
                                              colsample_bytree=0.8, tree_method='hist', random_state=s),
    'Small MLP':      lambda s: sc(MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=200, early_stopping=True,
                                                 random_state=s)),
}
def proba(m, X):                                        # probability of the positive class for every model family
    if hasattr(m, 'predict_proba'): return m.predict_proba(X)[:, 1]
    return 1 / (1 + np.exp(-m.decision_function(X)))
rows, fitted = [], {}
for name, make in models.items():
    aucs, aps, briers, secs = [], [], [], []
    for seed in range(5):
        m = make(seed); t0 = time.perf_counter(); m.fit(X_tr, y_tr); secs.append(time.perf_counter() - t0)
        p = proba(m, X_te); aucs.append(roc_auc_score(y_te, p)); aps.append(average_precision_score(y_te, p))
        briers.append(brier_score_loss(y_te, p))
    fitted[name] = m                                    # keep the last fitted model of each family for Step 6b
    rows.append([name, np.mean(aucs), np.std(aucs), np.mean(aps), np.mean(briers), np.mean(secs)])
res = pd.DataFrame(rows, columns=['model', 'ROC_AUC', 'sd', 'PR_AUC', 'Brier', 'train_s'])
res.to_csv('report/table_rq2.csv', index=False); print(res.round(4))

### Hình RQ2

- **Làm gì:** vẽ kết quả các mô hình, tô màu nhấn mô hình tốt nhất.
- **Kiểm tra:** số trên cột khớp với `report/table_rq2.csv`.

In [ ]:
import matplotlib.pyplot as plt
def bar_chart(labels, values, ylabel, path, highlight=None, fmt='{:.1f}'):
    fig, ax = plt.subplots(figsize=(6.5, 3.2))
    colors = ['#F4A261' if i == highlight else '#1B6B6D' for i in range(len(values))]
    bars = ax.bar([str(l) for l in labels], values, color=colors)
    ax.bar_label(bars, labels=[fmt.format(v) for v in values], fontweight='bold', padding=2, fontsize=8 if len(values) > 7 else 10)
    ax.set_ylabel(ylabel); ax.set_ylim(0, max(values) * 1.2)
    ax.spines[['top', 'right']].set_visible(False)          # no top and right spines
    if len(values) > 7: ax.tick_params(axis='x', labelsize=7, rotation=30); plt.setp(ax.get_xticklabels(), ha='right')
    fig.tight_layout(); fig.savefig(path, dpi=300); plt.show()

bar_chart(res.model, res.iloc[:, 1], res.columns[1], 'report/fig_rq2_models.png', highlight=int(res.iloc[:, 1].idxmax()), fmt='{:.2f}')

## Cổng khả thi: có nên xoay câu hỏi không

- **Làm gì:** so mô hình tốt nhất với mốc đơn giản và in ra một trong ba kết luận.
- **Vì sao:** nếu mô hình không hơn mốc hoặc bài toán quá dễ, nhóm xoay câu hỏi ngay ở đây thay vì cố chỉnh mô hình. Xoay câu hỏi nghĩa là đổi điều được hỏi cho khớp với điều dữ liệu trả lời được, không phải chọn lại số liệu cho đẹp.

**Các hướng xoay của đề tài này:**

1. Mô hình hơn quy tắc mưa trên cả bãi đã thấy và chưa thấy là kết quả dương: giữ trục cảnh báo trong ngày.
2. Nếu phản biện hỏi vì sao không dùng số đo tại chỗ: báo kết quả ba mức thông tin, số đo tại chỗ chỉ thêm rất ít.
3. Nếu muốn mục tiêu nghiêm hơn: đổi ngưỡng sang 200 đơn vị (mức kém) và lặp lại.

**Quyết định của nhóm (giữ câu hỏi hay xoay theo hướng nào, vì sao):** ...

In [ ]:
best = res.sort_values('ROC_AUC').iloc[-1]; spread = res.ROC_AUC.max() - res.ROC_AUC.min()
print('best model %s: ROC-AUC %.4f, PR-AUC %.4f | spread between models %.4f' % (best.model, best.ROC_AUC, best.PR_AUC, spread))
verdict = 'TOO EASY: every model is near the ceiling, so the comparison says little; explain why the task is easy or use a harder protocol (RQ3)' if best.ROC_AUC >= 0.995 else 'GOOD: there is a clear signal and room to compare models' if best.ROC_AUC >= 0.75 else 'WEAK: the signal is weak, pivot the target or the features (see the options above) and report this result honestly'
print(verdict); pd.Series({'best_model': best.model, 'ROC_AUC': best.ROC_AUC, 'PR_AUC': best.PR_AUC, 'spread': spread, 'verdict': verdict}).to_csv('report/table_feasibility_gate.csv')

## Bước 6b. Câu hỏi thứ ba (RQ3)

- **Làm gì:** Chấm hai quy tắc đơn giản và mười mô hình trên bãi đã thấy và bãi giữ riêng, huấn luyện lại bốn mô hình đại diện với ba mức thông tin và với cách chia ngẫu nhiên, rồi lập danh sách cảnh báo từ mô hình tốt nhất.
- **Vì sao:** đây là đóng góp chính của bài; mỗi bảng `report/table_rq3_*.csv` ứng với một ý trong mục Results.
- **Kiểm tra:** đọc từng bảng in ra và viết một câu kết luận có số cho mỗi bảng. Nếu kết quả trái dự kiến trong đề cương thì báo đúng như vậy và tìm nguyên nhân, không chỉnh code cho ra kết quả mong muốn.

In [ ]:
# Step 6b: RQ3, how far is the model above the rain rule that councils use, does it hold at sites it never saw, and what can be said before anyone goes to the beach with a probe?
from sklearn.model_selection import train_test_split
FAM = ['Logistic', 'Random Forest', 'LightGBM', 'Small MLP']; rows = [['Rule: rain in the last three days', roc_auc_score(y_te, test.rain_3d + test.rain_today), roc_auc_score(test_unseen[TARGET], test_unseen.rain_3d + test_unseen.rain_today)],
                                                                    ['Rule: unsafe rate of the site so far', roc_auc_score(y_te, test.site_unsafe_rate_before), roc_auc_score(test_unseen[TARGET], test_unseen.site_unsafe_rate_before)]]
for name, m in fitted.items(): rows.append([name, roc_auc_score(y_te, proba(m, X_te)), roc_auc_score(test_unseen[TARGET], proba(m, test_unseen[X_cols]))])      # (1) rules against models, seen and unseen sites
us = pd.DataFrame(rows, columns=['method', 'AUC_sites_in_training', 'AUC_sites_left_out']); us.to_csv('report/table_rq3_rules_and_unseen_sites.csv', index=False); print(us.round(3))
rows = []                                                              # (2) three levels of information: rain and calendar, plus site history, plus the probe readings taken with the sample
for name in FAM:
    row = [name]
    for cols in [RAIN_F, RAIN_F + SITE_F, X_cols]: row.append(roc_auc_score(y_te, proba(models[name](0).fit(train[cols], y_tr), test[cols])))
    rows.append(row)
lv = pd.DataFrame(rows, columns=['model', 'AUC_rain_and_calendar', 'AUC_plus_site_history', 'AUC_plus_probe_readings']); lv.to_csv('report/table_rq3_information_levels.csv', index=False); print(lv.round(3))
rows = []; allf = feat[~feat.site.isin(UNSEEN)]; a, b = train_test_split(allf, test_size=len(test) / len(allf), random_state=0, stratify=allf[TARGET]); a = cap(a, 60_000)      # (3) random split of all years
for name in FAM: rows.append([name, roc_auc_score(b[TARGET], proba(models[name](0).fit(a[X_cols], a[TARGET]), b[X_cols])), roc_auc_score(y_te, proba(fitted[name], X_te))])
sp = pd.DataFrame(rows, columns=['model', 'AUC_random_split', 'AUC_later_years']); sp.to_csv('report/table_rq3_split.csv', index=False); print(sp.round(3))

## Bước 6c. Kiểm tra bổ sung và bảng tham số

- **Làm gì:** chạy phép kiểm tra bổ sung của đề tài và lưu tham số mô hình.
- **Vì sao:** bảng này đi vào phần thảo luận và phụ lục của bài.
- **Kiểm tra:** `report/params_best.json` tồn tại.

In [ ]:
# Step 6c: a warning list for the daily beach report, calibration, strongest features, and parameters
best = res.sort_values('ROC_AUC').model.iloc[-1]; T = test.assign(p=np.clip(proba(fitted[best], X_te), 0, 1), rule=test.rain_3d + test.rain_today); rows = []
for share in [0.10, 0.20, 0.30]:
    for tag, col in [(best, 'p'), ('Rain rule', 'rule')]:
        t = T[col].quantile(1 - share); hit = T[col] >= t; rows.append([share, tag, int(hit.sum()), float(T.loc[hit, TARGET].mean()), float(T.loc[hit, TARGET].sum() / T[TARGET].sum())])
wl = pd.DataFrame(rows, columns=['share_of_samples_flagged', 'method', 'warnings', 'precision', 'recall']); wl.to_csv('report/table_rq3_warning_list.csv', index=False); print(best); print(wl.round(3))
T['bin'] = pd.cut(T.p, [0, .1, .2, .4, .6, 1.0001], labels=['0 to 0.1', '0.1 to 0.2', '0.2 to 0.4', '0.4 to 0.6', '0.6 to 1'], include_lowest=True)
cal = T.groupby('bin', observed=True).agg(samples=('p', 'size'), mean_predicted=('p', 'mean'), unsafe_share=(TARGET, 'mean')); cal.to_csv('report/table_rq3_calibration.csv'); print(cal.round(3))
from sklearn.inspection import permutation_importance
s = test.sample(min(4000, len(test)), random_state=0); imp = permutation_importance(fitted[best], s[X_cols], s[TARGET], scoring='roc_auc', n_repeats=3, random_state=0)
top = pd.DataFrame({'feature': X_cols, 'drop_in_ROC_AUC': imp.importances_mean}).sort_values('drop_in_ROC_AUC', ascending=False).head(10); top.to_csv('report/table_rq3_feature_importance.csv', index=False); print(top.round(4))
import json
json.dump({'best': best, 'unsafe_limit_cfu_per_100ml': 40, 'sites_left_out': sorted(UNSEEN), 'params': {k: str(v) for k, v in models[best](0).get_params().items()}}, open('report/params_best.json', 'w'), indent=2)

### Hình RQ3

- **Làm gì:** vẽ hai hình RQ3 từ các bảng `report/table_rq3_*.csv` bằng hàm `bar_chart` ở trên, theo mẫu Hình 4 và Hình 5 của đề cương.
- **Kiểm tra:** mỗi hình một ý, chữ tiếng Anh, số in đậm trên cột.

In [ ]:
import glob
for f in sorted(glob.glob('report/table_rq3_*.csv')):
    t = pd.read_csv(f); numc = [c for c in t.columns[1:] if pd.api.types.is_numeric_dtype(t[c])]
    print(f, t.shape)
    if 2 <= len(t) <= 14 and numc and t[numc[0]].notna().all() and t[numc[0]].min() >= 0 and t[numc[0]].max() > 0:
        v, lab, fm = nice(t[numc[0]], numc[0]); bar_chart(t.iloc[:, 0].astype(str), v, lab, f.replace('table_', 'fig_').replace('.csv', '.png'), highlight=int(v.argmax()), fmt=fm)

## Test case và chẩn đoán

- **Làm gì:** kiểm tra tự động các sản phẩm của notebook.
- **Kiểm tra:** mọi `assert` qua. Sinh viên A chạy lại notebook này và ghi vào ô Đối chứng.

In [ ]:
assert os.path.exists('report/table_rq2.csv') and os.path.exists('report/params_best.json')
assert len(glob.glob('report/table_rq3_*.csv')) >= 2, 'RQ3 tables missing'
assert res.iloc[:, 1].notna().all()
print(sorted(os.path.basename(f) for f in glob.glob('report/*')))
print('Notebook B: all checks passed')

### Đối chứng

| Người chạy lại | Ngày | Metric chính khớp | Ghi chú |
|---|---|---|---|
| | | | |

### Chẩn đoán

Nếu mô hình tốt nhất không hơn mốc đơn giản hoặc kết quả RQ3 trái dự kiến, ghi nguyên nhân có bằng chứng (thiếu đặc trưng, dữ liệu ít, lệch giữa hai tập) vào `report/table_diagnosis.csv`.

### Báo cáo tiến độ (dán vào kênh nhóm trước buổi báo cáo)

1. Việc đã xong, kèm tên file và số dòng, bảng, hình:
2. Con số chính mới có:
3. Vướng mắc và điều đã thử:
4. Việc sẽ làm đến lần báo cáo sau, ai làm:
5. Prompt AI đáng chú ý và Human Delta: